# Evaluate Probabilities — Base Case
Extract P(correct) for each pair, plot position-invariant and position-split heatmaps.

In [ ]:
import os
from pathlib import Path
os.chdir(Path("/mnt/smb/locker/abbott-locker/Luke/Mnist") / "Mnist v3")
!pwd

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

from TransitiveTestDataset import TransitiveTestDataset
from TransitiveTrainDataset import TransitiveTrainDataset

## Parameters

In [ ]:
MODEL_PATH = "ti_cnn.pt"
N_ITEMS = 8
SAMPLES_PER_PAIR = 2000
DATA_SEED = 42
BATCH_SIZE = 1000

## Model

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(19968, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

## Load Model & Data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Net().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"Loaded model from {MODEL_PATH}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('./data', train=False, transform=transform)

train_dataset = TransitiveTrainDataset(mnist_train, n=N_ITEMS, samples_per_pair=SAMPLES_PER_PAIR, seed=DATA_SEED)
test_dataset = TransitiveTestDataset(mnist_test, n=N_ITEMS, samples_per_pair=SAMPLES_PER_PAIR, seed=DATA_SEED)

print(f"Train pairs: {train_dataset.pairs}")
print(f"Test pairs: {test_dataset.pairs}")

## Extract Probabilities

In [ ]:
def extract_probabilities(model, device, dataset, batch_size=1000):
    model.eval()

    probs_by_pair = defaultdict(list)
    probs_by_pair_left = defaultdict(list)
    probs_by_pair_right = defaultdict(list)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    sample_idx = 0
    with torch.no_grad():
        for stimuli, labels in loader:
            stimuli, labels = stimuli.to(device), labels.to(device)
            output = model(stimuli)
            probs = torch.exp(output)

            for j in range(stimuli.size(0)):
                i = sample_idx + j
                sample_i = i // 2
                is_flipped = i % 2
                winner_digit, loser_digit, _, _ = dataset.samples[sample_i]

                p_correct = probs[j, labels[j]].item()

                probs_by_pair[(winner_digit, loser_digit)].append(p_correct)

                if is_flipped == 0:
                    probs_by_pair_left[(winner_digit, loser_digit)].append(p_correct)
                else:
                    probs_by_pair_right[(winner_digit, loser_digit)].append(p_correct)

            sample_idx += stimuli.size(0)

    pair_probs = {k: np.mean(v) for k, v in probs_by_pair.items()}
    pair_probs_left = {k: np.mean(v) for k, v in probs_by_pair_left.items()}
    pair_probs_right = {k: np.mean(v) for k, v in probs_by_pair_right.items()}

    return pair_probs, pair_probs_left, pair_probs_right

## Plotting Functions

In [ ]:
def plot_position_invariant(pair_probs, title="P(correct) by pair", save_path=None):
    pairs = sorted(pair_probs.keys())
    all_items = sorted(set([p[0] for p in pairs] + [p[1] for p in pairs]))
    n = max(all_items) + 1

    matrix = np.full((n, n), np.nan)
    for (w, l), p in pair_probs.items():
        matrix[w, l] = p

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')

    for i in range(n):
        for j in range(n):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', fontsize=9,
                        color='black' if 0.3 < matrix[i, j] < 0.7 else 'white')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xlabel('Loser')
    ax.set_ylabel('Winner')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='P(correct)')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_position_split(pair_probs_left, pair_probs_right, title="P(correct) by position", save_path=None):
    all_pairs = sorted(set(list(pair_probs_left.keys()) + list(pair_probs_right.keys())))
    all_items = sorted(set([p[0] for p in all_pairs] + [p[1] for p in all_pairs]))
    n = max(all_items) + 1

    matrix_left = np.full((n, n), np.nan)
    matrix_right = np.full((n, n), np.nan)

    for (w, l), p in pair_probs_left.items():
        matrix_left[w, l] = p
    for (w, l), p in pair_probs_right.items():
        matrix_right[w, l] = p

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, matrix, subtitle in [(ax1, matrix_left, 'Winner on LEFT'),
                                  (ax2, matrix_right, 'Winner on RIGHT')]:
        im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
        for i in range(n):
            for j in range(n):
                if not np.isnan(matrix[i, j]):
                    ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', fontsize=9,
                            color='black' if 0.3 < matrix[i, j] < 0.7 else 'white')
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xlabel('Loser')
        ax.set_ylabel('Winner')
        ax.set_title(subtitle)
        plt.colorbar(im, ax=ax, label='P(correct)')

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_position_bias(pair_probs_left, pair_probs_right, title="Position bias (LEFT - RIGHT)", save_path=None):
    all_pairs = sorted(set(list(pair_probs_left.keys()) + list(pair_probs_right.keys())))
    all_items = sorted(set([p[0] for p in all_pairs] + [p[1] for p in all_pairs]))
    n = max(all_items) + 1

    matrix_diff = np.full((n, n), np.nan)
    for (w, l) in all_pairs:
        if (w, l) in pair_probs_left and (w, l) in pair_probs_right:
            matrix_diff[w, l] = pair_probs_left[(w, l)] - pair_probs_right[(w, l)]

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    im = ax.imshow(matrix_diff, cmap='RdBu', vmin=-0.5, vmax=0.5, origin='upper')

    for i in range(n):
        for j in range(n):
            if not np.isnan(matrix_diff[i, j]):
                ax.text(j, i, f'{matrix_diff[i, j]:+.2f}', ha='center', va='center', fontsize=9)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xlabel('Loser')
    ax.set_ylabel('Winner')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='P(correct|left) - P(correct|right)')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def print_table(pair_probs, pair_probs_left, pair_probs_right):
    print(f"{'Pair':>10} | {'P(correct)':>10} | {'Winner LEFT':>12} | {'Winner RIGHT':>12} | {'Bias (L-R)':>10}")
    print("-" * 65)
    for pair in sorted(pair_probs.keys()):
        p = pair_probs[pair]
        pl = pair_probs_left.get(pair, float('nan'))
        pr = pair_probs_right.get(pair, float('nan'))
        bias = pl - pr
        print(f"  {pair[0]} > {pair[1]}   |   {p:.4f}   |     {pl:.4f}   |      {pr:.4f}   |   {bias:+.4f}")

## Test Set

In [ ]:
print("=== Test Set (non-adjacent pairs) ===")
pair_probs, pair_probs_left, pair_probs_right = extract_probabilities(model, device, test_dataset, BATCH_SIZE)
print_table(pair_probs, pair_probs_left, pair_probs_right)

In [ ]:
plot_position_invariant(pair_probs,
                        title="Test Set — P(correct) by pair",
                        save_path="test_probs_invariant.png")

In [ ]:
plot_position_split(pair_probs_left, pair_probs_right,
                    title="Test Set — P(correct) by winner position",
                    save_path="test_probs_split.png")

In [ ]:
plot_position_bias(pair_probs_left, pair_probs_right,
                   title="Test Set — Position bias",
                   save_path="test_probs_bias.png")

## Training Set

In [ ]:
print("=== Training Set (adjacent pairs) ===")
pair_probs_tr, pair_probs_left_tr, pair_probs_right_tr = extract_probabilities(model, device, train_dataset, BATCH_SIZE)
print_table(pair_probs_tr, pair_probs_left_tr, pair_probs_right_tr)

In [ ]:
plot_position_invariant(pair_probs_tr,
                        title="Train Set — P(correct) by pair",
                        save_path="train_probs_invariant.png")

In [ ]:
plot_position_split(pair_probs_left_tr, pair_probs_right_tr,
                    title="Train Set — P(correct) by winner position",
                    save_path="train_probs_split.png")

In [ ]:
plot_position_bias(pair_probs_left_tr, pair_probs_right_tr,
                   title="Train Set — Position bias",
                   save_path="train_probs_bias.png")